In [ ]:
# 基于出租车原始 GPS 轨迹数据，通过载客状态的时序变化规律，识别乘客上下车事件，最终生成标准化的出行 OD（起点 - 终点）数据表
# 完成时间：2026-7-3~2026-7-5
# 练习内容：geopandas基础数据提取处理，初步培养时空轨迹数据处理思维

读取数据

In [1]:
import pandas as pd
# 读取数据(header=None告诉 pandas，文件第一行不是列名，所有行都是数据，列名默认用0、1、2...数字编号)
data = pd.read_csv(r'E:/GIS_Learning/data/taxiData-sample.csv', header=None)
# 给数据命名列
data.columns = ['VehicleNum', 'Stime', 'Lng', 'Lat', 'OpenStatus', 'Speed']
# 验证：打印行数和列名，确认data创建成功
print("数据行数：", data.shape[0])
print("所有列名：", data.columns.tolist())
data.head()

数据行数： 1601307
所有列名： ['VehicleNum', 'Stime', 'Lng', 'Lat', 'OpenStatus', 'Speed']


,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed
0,22271,22:54:04,114.167000,22.718399,0,0
1,22271,18:26:26,114.190598,22.647800,0,4
2,22271,18:35:18,114.201401,22.649700,0,0
3,22271,16:02:46,114.233498,22.725901,0,24
4,22271,21:41:17,114.233597,22.720900,0,19


基础的数据操作(DataFrame和Series)

In [2]:
# 显示数据的前5行
data.head(5)

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed
0,22271,22:54:04,114.167000,22.718399,0,0
1,22271,18:26:26,114.190598,22.647800,0,4
2,22271,18:35:18,114.201401,22.649700,0,0
3,22271,16:02:46,114.233498,22.725901,0,24
4,22271,21:41:17,114.233597,22.720900,0,19


In [3]:
# 查看格式（DataFrame是由一列一列组成的表，每一列为一个series系列）
type(data)

pandas.core.frame.DataFrame

In [4]:
# 提取经度这一列
data['Lng']

0          114.167000
1          114.190598
2          114.201401
3          114.233498
4          114.233597
              ...    
1601302    114.160149
1601303    114.164551
1601304    114.168015
1601305    114.168015
1601306    114.170647
Name: Lng, Length: 1601307, dtype: float64

In [5]:
# 提取两列
data[['Lng','Lat']]

,Lng,Lat
0,114.167000,22.718399
1,114.190598,22.647800
2,114.201401,22.649700
3,114.233498,22.725901
4,114.233597,22.720900
...,...,...
1601302,114.160149,22.606934
1601303,114.164551,22.605118
1601304,114.168015,22.606083
1601305,114.168015,22.606083


数据筛选

In [6]:
# 筛选车牌照为22271的所有数据
data[data['VehicleNum']==22271]
# 会生成一个布尔型 Series：每一行对应一个 True/False，车辆编号等于 22271 则为 True，否则为 False

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed
0,22271,22:54:04,114.167000,22.718399,0,0
1,22271,18:26:26,114.190598,22.647800,0,4
2,22271,18:35:18,114.201401,22.649700,0,0
3,22271,16:02:46,114.233498,22.725901,0,24
4,22271,21:41:17,114.233597,22.720900,0,19
...,...,...,...,...,...,...
1432,22271,11:31:09,114.279404,22.739000,0,0
1433,22271,11:33:09,114.279404,22.739000,0,0
1434,22271,10:43:56,114.279503,22.739000,0,0
1435,22271,10:51:48,114.279503,22.739201,0,0


In [7]:
# 删除所有编号为 22271 的数据，保留其他车辆的数据
data[-((data['VehicleNum']==22271)|(data['VehicleNum']==23873))]
# -表示逻辑取反：True 变 False，False 变 True
# data.drop()只在想删除某一列的时候用，为避免bug还是少用

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed
1437,35807,01:53:46,113.809898,22.626801,0,0
1438,35807,01:43:46,113.813301,22.623600,0,0
1439,35807,01:14:15,113.847000,22.594700,0,41
1440,35807,02:01:41,113.852501,22.625700,0,22
1441,35807,01:01:59,113.897003,22.551901,0,42
...,...,...,...,...,...,...
1594428,29258,12:46:07,114.032150,22.630030,1,0
1594429,29258,12:16:36,114.032249,22.614902,1,20
1594430,29258,12:16:06,114.032684,22.614632,1,0
1594431,29258,12:37:37,114.036285,22.610331,1,75


获取/删除/定义DataFrame的某一列（用iloc(按表目前排列的顺序取)，不能用loc(按index取)
因为很多时候我们做完筛选、排序等操作，表就不是按index来排列，用loc取列就会取错列，或者直接报错（没有这个index））

In [8]:
# 获取列'Stime',注意，此操作不会影响到data，你在操作后必须将得到的表重新赋值给data才有影响
data[['Stime']].head(5)

,Stime
0,22:54:04
1,18:26:26
2,18:35:18
3,16:02:46
4,21:41:17


In [9]:
# 定义列'Speed1'为Speed列的两倍,注意，此操作会影响到data
data['Speed1']=data['Speed']*2
# 或者(更规范)
data.loc[:,'Speed1']=data['Speed']*2
data.head(5)

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed,Speed1
0,22271,22:54:04,114.167000,22.718399,0,0,0
1,22271,18:26:26,114.190598,22.647800,0,4,8
2,22271,18:35:18,114.201401,22.649700,0,0,0
3,22271,16:02:46,114.233498,22.725901,0,24,48
4,22271,21:41:17,114.233597,22.720900,0,19,38


In [10]:
# 删除列'Stime',注意，此操作不会影响到data，你在操作后必须将得到的表重新赋值给data才有影响
data.drop(['Stime'],axis=1).head(5) 
# drop()用于删除列或行，axis=1 表示按列删除；axis=0表示按行删除（不建议把删除结果重新赋值给data）

,VehicleNum,Lng,Lat,OpenStatus,Speed,Speed1
0,22271,114.167000,22.718399,0,0,0
1,22271,114.190598,22.647800,0,4,8
2,22271,114.201401,22.649700,0,0,0
3,22271,114.233498,22.725901,0,24,48
4,22271,114.233597,22.720900,0,19,38


In [11]:
# 把删除结果重新赋值给data，永久删除Speed1列
data = data.drop(['Speed1'],axis=1)
data.head(5)

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed
0,22271,22:54:04,114.167000,22.718399,0,0
1,22271,18:26:26,114.190598,22.647800,0,4
2,22271,18:35:18,114.201401,22.649700,0,0
3,22271,16:02:46,114.233498,22.725901,0,24
4,22271,21:41:17,114.233597,22.720900,0,19


In [12]:
#获取Stime列的第4行数据
data['Stime'].iloc[3] # iloc是按位置序号取数，从 0 开始计数

'16:02:46'

数据清洗

In [13]:
# 将数据排序,并把排序后的数据赋值给原来的数据
# 核心作用：保证每辆车的 GPS 轨迹按时间顺序排列，以便后续用shift()对比前后行的状态变化
data = data.sort_values(by = ['VehicleNum','Stime'])
# sort_values()：按指定列对整张表排序
# by = ['VehicleNum','Stime']：先按车辆编号排序，把同一辆车的所有数据排在一起；同一辆车内部，再按时间从小到大排序
data.head(5)
# 重新赋值以生效

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed
39,22271,00:00:49,114.266502,22.728201,0,0
397,22271,00:01:48,114.266502,22.728201,0,0
1413,22271,00:02:47,114.266502,22.728201,0,0
244,22271,00:03:46,114.266502,22.728201,0,0
247,22271,00:04:45,114.268898,22.729500,0,11


当OpenStatus由0->1说明乘客上车，由1->0说明乘客下车

异常处理（前后都是0，突然有一条数据变成1，或者前后都是1，突然变成0）

In [14]:
# shift()函数能够将数据按顺序后移一位
data2=data.iloc[2460:2480]
data2['OpenStatus'].shift()
data2

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed
1550423,22334,06:27:51,114.036018,22.519051,0,19
1549900,22334,06:28:11,114.035248,22.517817,0,25
1549352,22334,06:28:31,114.035149,22.517616,0,5
1550422,22334,06:28:51,114.034569,22.517967,0,0
1547256,22334,06:29:11,114.032349,22.518984,0,21
1547418,22334,06:29:31,114.031250,22.519550,0,22
1547419,22334,06:29:51,114.031250,22.519550,0,0
1547764,22334,06:30:11,114.031250,22.519550,0,0
1550417,22334,06:30:19,114.031250,22.519550,1,0
1547416,22334,06:30:39,114.030098,22.520317,1,18


In [15]:
"""
这里，把上述异常数据清洗出来
用到的条件是：
1.后一位和前一位相等
2.但是后一位与中间一位不等
3.前一条数据，后一条数据的车牌相等
4.中间一条数据，后一条数据的车牌相等

各条件之间的交集可以用
data[(条件1)&(条件2)]
各条件之间的并集可以用
data[(条件1)|(条件2)]


data =
"""

'\n这里，把上述异常数据清洗出来\n用到的条件是：\n1.后一位和前一位相等\n2.但是后一位与中间一位不等\n3.前一条数据，后一条数据的车牌相等\n4.中间一条数据，后一条数据的车牌相等\n\n各条件之间的交集可以用\ndata[(条件1)&(条件2)]\n各条件之间的并集可以用\ndata[(条件1)|(条件2)]\n\n\ndata =\n'

In [16]:
# 识别「前后行状态一致、只有当前行突然跳变」的异常数据（比如连续空车中突然出现 1 条载客记录）
data = data[-((data['OpenStatus'].shift(-1) == data['OpenStatus'].shift())& # 下一行状态 = 上一行状态
(data['OpenStatus'].shift(-1) != data['OpenStatus'])& # 下一行状态 ≠ 当前行状态
(data['VehicleNum'].shift(-1) == data['VehicleNum'].shift())& # 前后行是同一辆车
(data['VehicleNum'].shift(-1) == data['VehicleNum']))] # 当前行与下一行是同一辆车

识别OD(乘客上下车的状态变化识别)

In [17]:
# 让这几个字段的下一条数据赋值给新的字段，在字段名加个1，代表后面一条数据的值
data.loc[:,'OpenStatus1'] = data['OpenStatus'].shift(-1)
data.loc[:,'VehicleNum1'] = data['VehicleNum'].shift(-1)
data.loc[:,'Lng1'] = data['Lng'].shift(-1)
data.loc[:,'Lat1'] = data['Lat'].shift(-1)
data.loc[:,'Stime1'] = data['Stime'].shift(-1)
# shift(-1)：将整列数据向上移动 1 行，相当于把下一行的数值放到当前行的新列中
# 新列名加后缀1，统一表示 “下一行的对应字段”

# 计算状态变化(状态变化 = 下一行状态 - 当前行状态)
data.loc[:,'StatusChange'] = data['OpenStatus1']-data['OpenStatus']

data.head(5)

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed,OpenStatus1,VehicleNum1,Lng1,Lat1,Stime1,StatusChange
39,22271,00:00:49,114.266502,22.728201,0,0,0.0,22271.0,114.266502,22.728201,00:01:48,0.0
397,22271,00:01:48,114.266502,22.728201,0,0,0.0,22271.0,114.266502,22.728201,00:02:47,0.0
1413,22271,00:02:47,114.266502,22.728201,0,0,0.0,22271.0,114.266502,22.728201,00:03:46,0.0
244,22271,00:03:46,114.266502,22.728201,0,0,0.0,22271.0,114.268898,22.729500,00:04:45,0.0
247,22271,00:04:45,114.268898,22.729500,0,11,0.0,22271.0,114.272003,22.731199,00:05:44,0.0


将上车状态整理成OD

In [18]:
# 保留StatusChange字段为1或者-1的数据，因此要把这些数据筛出来
# 不过，还得加一个条件，就是这条数据和下一条数据的车辆ID必须是相同的

In [19]:
# 筛选状态变化的点
data = data[((data['StatusChange'] == 1)|(data['StatusChange'] == -1)) # 只保留发生了上车或下车事件的行，剔除所有状态无变化的冗余行驶数据
# 保证同车
&(data['VehicleNum'] == data['VehicleNum1'])] # 排除两辆车的边界行（前一辆车最后一行 + 后一辆车第一行），避免跨车辆错误配对

data.head(5)

,VehicleNum,Stime,Lng,Lat,OpenStatus,Speed,OpenStatus1,VehicleNum1,Lng1,Lat1,Stime1,StatusChange
1548741,22334,00:00:52,114.111130,22.576750,1,13,0.0,22334.0,114.111130,22.576750,00:01:04,-1.0
1548351,22334,00:07:44,114.080498,22.554182,0,11,1.0,22334.0,114.080498,22.554182,00:07:57,1.0
1549620,22334,00:17:58,114.084915,22.540850,1,2,0.0,22334.0,114.084915,22.540850,00:18:16,-1.0
1547182,22334,00:18:56,114.084915,22.540850,0,0,1.0,22334.0,114.084915,22.540850,00:19:05,1.0
1547627,22334,00:44:47,114.056236,22.633383,1,3,0.0,22334.0,114.056236,22.633383,00:44:52,-1.0


In [20]:
# 1. 重命名原始经纬度为起点字段（明确标识这是上车点的位置，与后续下车终点字段对应，语义更清晰）
data = data.rename(columns={'Lng':'SLng', 'Lat':'SLat'})

# 2. 按车辆分组，计算状态变化列
data['StatusChange'] = data.groupby('VehicleNum')['OpenStatus'].diff() # diff()：pandas 内置函数，效果等价于「当前行 - 上一行」

# 3. 按车辆分组，匹配下一行的终点信息
data['ELng'] = data.groupby('VehicleNum')['SLng'].shift(-1)
data['ELat'] = data.groupby('VehicleNum')['SLat'].shift(-1)
data['Etime'] = data.groupby('VehicleNum')['Stime'].shift(-1)
# 筛选后的状态变化行是 “上车→下车→上车→下车” 交替排列，每一个上车行的下一行，正好是对应行程的下车行；用shift(-1)平移后，上车行就能同时携带起点 + 终点的完整信息

# 4. 筛选出状态变化为1的上车行（完整OD对）
data = data[data['StatusChange'] == 1]

# 5. 删除不需要的辅助列
data = data.drop('StatusChange', axis=1)

# 查看结果
data.head()

,VehicleNum,Stime,SLng,SLat,OpenStatus,Speed,OpenStatus1,VehicleNum1,Lng1,Lat1,Stime1,ELng,ELat,Etime
1549620,22334,00:17:58,114.084915,22.540850,1,2,0.0,22334.0,114.084915,22.540850,00:18:16,114.084915,22.540850,00:18:56
1547627,22334,00:44:47,114.056236,22.633383,1,3,0.0,22334.0,114.056236,22.633383,00:44:52,114.091637,22.543200,02:38:35
1547892,22334,02:46:52,114.093498,22.554382,1,20,0.0,22334.0,114.093536,22.554382,02:47:04,114.038818,22.553232,03:58:46
1549189,22334,04:13:57,114.052299,22.604366,1,7,0.0,22334.0,114.052216,22.602118,04:23:07,114.031250,22.519550,06:30:11
1550681,22334,06:41:19,114.067886,22.521299,1,2,0.0,22334.0,114.067886,22.521299,06:41:20,114.063034,22.534634,06:49:40


In [21]:
# 只保留OD核心字段，列名和49运行后的结果保持一致
data = data[['VehicleNum', 'Stime', 'SLng', 'SLat', 'Etime', 'ELng', 'ELat']]
data.head()

,VehicleNum,Stime,SLng,SLat,Etime,ELng,ELat
1549620,22334,00:17:58,114.084915,22.540850,00:18:56,114.084915,22.540850
1547627,22334,00:44:47,114.056236,22.633383,02:38:35,114.091637,22.543200
1547892,22334,02:46:52,114.093498,22.554382,03:58:46,114.038818,22.553232
1549189,22334,04:13:57,114.052299,22.604366,06:30:11,114.031250,22.519550
1550681,22334,06:41:19,114.067886,22.521299,06:49:40,114.063034,22.534634


In [22]:
# 保存文件
data.to_csv(r'E:\GIS_Learning\data\TaxiOD-Sample-finish.csv',index = None)